# ERA5-Land daily Tmax/Tmin extraction — Kakamega, Kilifi, Nyeri 

Downloads ERA5-Land daily maximum/minimum 2m temperature (already daily-resolution from the CDS API), 
clips each day's raster to each county polygon, and produces **two output types**:

1. A long-format daily CSV (`date, county, tmax_c, tmin_c`) — the county spatial mean per day.
2. One **GeoTIFF per day, per variable, per county** (e.g. `Nyeri/tmax/Nyeri_tmax_20200105.tif`), 
   so you keep the full clipped raster for each day, not just the county-mean number.

That's roughly 6 years × 365 days × 3 counties × 2 variables ≈ 13,000 GeoTIFFs — expect this to take a while and use 
noticeable disk space; each is small (single county extent) but they add up.

Resumable: skips year-months already fully written to the CSV, and skips individual day TIFFs that already exist on disk 
(so a partial re-run backfills only what's missing).

In [ ]:
import os
import time
import traceback
import cdsapi
import xarray as xr
import rioxarray
import geopandas as gpd
import pandas as pd

output_base = r"E:\ERA5_LAND_TEMP"
shp_path = r"C:\Users\locateIT1\Downloads\ken_admin1.shp"
target_counties = ['Nyeri', 'Kakamega', 'Kilifi']

years = range(2020, 2026)   # 2020 to 2025 inclusive
months = range(1, 13)

csv_path = os.path.join(output_base, "daily_tmax_tmin_2020_2025.csv")
log_path = os.path.join(output_base, "daily_extraction_log.txt")

os.makedirs(output_base, exist_ok=True)

def log_message(msg):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = "[" + timestamp + "] " + msg
    print(line)
    with open(log_path, "a") as f:
        f.write(line + "\n")

## Helper: already-processed check
Reads the CSV so far and tells us whether a given year-month is fully covered for all target counties 
(so we can skip re-downloading it on a re-run).

In [ ]:
def load_existing_csv():
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path, parse_dates=['date'])
    return pd.DataFrame(columns=['date', 'county', 'tmax_c', 'tmin_c'])

def month_already_done(existing_df, yr, mo):
    if existing_df.empty:
        return False
    mask = (existing_df['date'].dt.year == yr) & (existing_df['date'].dt.month == mo)
    covered_counties = set(existing_df.loc[mask, 'county'].unique())
    return set(target_counties).issubset(covered_counties)

## Main extraction loop
For each year-month: download ERA5-Land daily max/min t2m (one value per day already, via `derived-era5-land-daily-statistics`), 
clip+average per county per day, append rows to the CSV, delete the raw .nc files, and log progress.

In [ ]:
counties = gpd.read_file(shp_path).to_crs("EPSG:4326")
c = cdsapi.Client()

for yr in years:
    for mo in months:
        yr_str = str(yr)
        mo_str = str(mo).zfill(2)

        existing_df = load_existing_csv()
        if month_already_done(existing_df, yr, mo):
            log_message("SKIP " + yr_str + "-" + mo_str + " (already in CSV)")
            continue

        max_filename = "t2m_max_" + yr_str + mo_str + ".nc"
        min_filename = "t2m_min_" + yr_str + mo_str + ".nc"
        download_path_max = os.path.join(output_base, max_filename)
        download_path_min = os.path.join(output_base, min_filename)

        try:
            log_message("START " + yr_str + "-" + mo_str)

            days_in_month = [str(d).zfill(2) for d in range(1, 32)]

            c.retrieve(
                'derived-era5-land-daily-statistics',
                {
                    'variable': '2m_temperature',
                    'year': yr_str,
                    'month': mo_str,
                    'day': days_in_month,
                    'daily_statistic': 'daily_maximum',
                    'time_zone': 'utc+00:00',
                    'frequency': '1_hourly',
                    'area': [5.5, 33.5, -5, 42.5],
                    'format': 'netcdf',
                },
                download_path_max
            )

            c.retrieve(
                'derived-era5-land-daily-statistics',
                {
                    'variable': '2m_temperature',
                    'year': yr_str,
                    'month': mo_str,
                    'day': days_in_month,
                    'daily_statistic': 'daily_minimum',
                    'time_zone': 'utc+00:00',
                    'frequency': '1_hourly',
                    'area': [5.5, 33.5, -5, 42.5],
                    'format': 'netcdf',
                },
                download_path_min
            )

            ds_max = xr.open_dataset(download_path_max)
            ds_min = xr.open_dataset(download_path_min)

            ds_max = ds_max.rename({'longitude': 'x', 'latitude': 'y', 'valid_time': 'time'}).rio.write_crs("EPSG:4326", inplace=False)
            ds_min = ds_min.rename({'longitude': 'x', 'latitude': 'y', 'valid_time': 'time'}).rio.write_crs("EPSG:4326", inplace=False)


            rows = []
            for county_name in target_counties:
                county_geom = counties[counties['adm1_name'] == county_name].geometry

                clipped_max_c = (ds_max['t2m'].rio.clip(county_geom, counties.crs, drop=True) - 273.15).rio.write_crs("EPSG:4326", inplace=False)
                clipped_min_c = (ds_min['t2m'].rio.clip(county_geom, counties.crs, drop=True) - 273.15).rio.write_crs("EPSG:4326", inplace=False)

                tmax_series = clipped_max_c.mean(dim=['y', 'x'], skipna=True).to_series()
                tmin_series = clipped_min_c.mean(dim=['y', 'x'], skipna=True).to_series()

                for dt, val in tmax_series.items():
                    rows.append({'date': pd.Timestamp(dt).normalize(), 'county': county_name, 'tmax_c': val, 'tmin_c': None})
                for dt, val in tmin_series.items():
                    rows.append({'date': pd.Timestamp(dt).normalize(), 'county': county_name, 'tmax_c': None, 'tmin_c': val})

                # --- write one GeoTIFF per day, per variable, per county ---
                county_tif_dir_max = os.path.join(output_base, county_name, "tmax")
                county_tif_dir_min = os.path.join(output_base, county_name, "tmin")
                os.makedirs(county_tif_dir_max, exist_ok=True)
                os.makedirs(county_tif_dir_min, exist_ok=True)

                for t in clipped_max_c['time'].values:
                    day_str = pd.Timestamp(t).strftime('%Y%m%d')
                    max_tif_path = os.path.join(county_tif_dir_max, county_name + "_tmax_" + day_str + ".tif")
                    if not os.path.exists(max_tif_path):
                        clipped_max_c.sel(time=t).rio.to_raster(max_tif_path)

                for t in clipped_min_c['time'].values:
                    day_str = pd.Timestamp(t).strftime('%Y%m%d')
                    min_tif_path = os.path.join(county_tif_dir_min, county_name + "_tmin_" + day_str + ".tif")
                    if not os.path.exists(min_tif_path):
                        clipped_min_c.sel(time=t).rio.to_raster(min_tif_path)

            month_df = pd.DataFrame(rows)
            month_df = month_df.groupby(['date', 'county'], as_index=False).agg({
                'tmax_c': 'max',
                'tmin_c': 'min'
            })

            write_header = not os.path.exists(csv_path)
            month_df.sort_values(['county', 'date']).to_csv(csv_path, mode='a', header=write_header, index=False)

            ds_max.close()
            ds_min.close()

            for f_path in [download_path_max, download_path_min]:
                if os.path.exists(f_path):
                    try:
                        os.remove(f_path)
                    except PermissionError:
                        log_message("WARNING: could not delete " + f_path + " (file locked)")

            log_message("DONE " + yr_str + "-" + mo_str)

        except Exception as e:
            log_message("ERROR " + yr_str + "-" + mo_str + ": " + str(e))
            log_message(traceback.format_exc())
            continue

log_message("ALL MONTHS PROCESSED")
print("Loop finished. Daily CSV at:", csv_path)

## Sanity check
Load the CSV and plot Nyeri's daily tmax/tmin, then re-open one saved daily GeoTIFF to confirm both outputs look right.

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(csv_path, parse_dates=['date'])
nyeri = df[df['county'] == 'Nyeri'].sort_values('date')

plt.figure(figsize=(12, 4))
plt.plot(nyeri['date'], nyeri['tmax_c'], label='Tmax (°C)', color='red', linewidth=0.7)
plt.plot(nyeri['date'], nyeri['tmin_c'], label='Tmin (°C)', color='blue', linewidth=0.7)
plt.title("Nyeri — Daily Tmax/Tmin (ERA5-Land, county spatial mean)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.tight_layout()
plt.show()

df.head()

In [ ]:
# Re-open one saved daily GeoTIFF to confirm the raster output looks right
sample_max = rioxarray.open_rasterio(os.path.join(output_base, "Nyeri", "tmax", "Nyeri_tmax_20200105.tif"))
sample_min = rioxarray.open_rasterio(os.path.join(output_base, "Nyeri", "tmin", "Nyeri_tmin_20200105.tif"))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sample_max.plot(ax=axes[0], cmap='Reds')
axes[0].set_title("Nyeri — Tmax (°C) — 2020-01-05")
sample_min.plot(ax=axes[1], cmap='Blues')
axes[1].set_title("Nyeri — Tmin (°C) — 2020-01-05")
plt.tight_layout()
plt.show()